# 04. 분석용 mart 만들기

stg 테이블을 기반으로 분석용 mart 4개를 생성하고 점검 결과를 Markdown으로 저장합니다.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "seoul_bike.db"
SQL_DIR = PROJECT_ROOT / "sql"
REPORT_DIR = PROJECT_ROOT / "reports"

if not DB_PATH.exists() and Path("/mnt/data/seoul_bike.db").exists():
    PROJECT_ROOT = Path("/mnt/data")
    DATA_DIR = PROJECT_ROOT
    DB_PATH = PROJECT_ROOT / "seoul_bike.db"
    SQL_DIR = PROJECT_ROOT
    REPORT_DIR = PROJECT_ROOT

SQL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
print("DB:", DB_PATH)

DB: d:\seoul-bike-aarrr-analysis\data\seoul_bike.db


In [3]:
required_stg_tables = {
    "stg_signup_month",
    "stg_ride_month",
    "stg_station_snapshot",
    "stg_station_usage_month",
    "stg_month_station_snapshot_map"
}

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)

missing = required_stg_tables - set(tables["name"])
if missing:
    raise ValueError(f"필수 stg 테이블이 없습니다: {missing}")
else:
    print("필수 stg 테이블 확인 완료")

필수 stg 테이블 확인 완료


In [14]:
mart_sql_path = SQL_DIR / "04_create_mart_tables.sql"

if mart_sql_path.exists():
    sql = mart_sql_path.read_text(encoding="utf-8")
else:
    sql = r'''-- 04_create_mart_tables.sql
-- SQLite 기준 mart 테이블 생성 스크립트

DROP TABLE IF EXISTS mart_signup_segment_month;
DROP TABLE IF EXISTS mart_ride_segment_month;
DROP TABLE IF EXISTS mart_growth_month;
DROP TABLE IF EXISTS mart_station_efficiency_month;

CREATE TABLE mart_signup_segment_month AS
SELECT
    month_ym,
    month_date,
    gender,
    age_band,
    SUM(signup_cnt) AS signup_cnt
FROM stg_signup_month
WHERE month_ym BETWEEN 202401 AND 202512
GROUP BY month_ym, month_date, gender, age_band;

CREATE TABLE mart_ride_segment_month AS
SELECT
    month_ym,
    month_date,
    gender,
    age_band,
    pass_group AS ride_type,
    SUM(ride_cnt) AS ride_cnt,
    SUM(distance_m) AS total_distance_m,
    SUM(duration_min) AS total_duration_min,
    CASE WHEN SUM(ride_cnt) > 0 THEN 1.0 * SUM(distance_m) / SUM(ride_cnt) ELSE NULL END AS avg_distance_m_per_ride,
    CASE WHEN SUM(ride_cnt) > 0 THEN 1.0 * SUM(duration_min) / SUM(ride_cnt) ELSE NULL END AS avg_duration_min_per_ride
FROM stg_ride_month
WHERE month_ym BETWEEN 202401 AND 202512
GROUP BY month_ym, month_date, gender, age_band, pass_group;

CREATE TABLE mart_growth_month AS
WITH signup AS (
    SELECT
        month_ym,
        month_date,
        gender,
        age_band,
        SUM(signup_cnt) AS signup_cnt
    FROM stg_signup_month
    WHERE month_ym BETWEEN 202401 AND 202512
    GROUP BY month_ym, month_date, gender, age_band
),
ride AS (
    SELECT
        month_ym,
        month_date,
        gender,
        age_band,
        SUM(CASE WHEN pass_group = 'SUBSCRIPTION' THEN ride_cnt ELSE 0 END) AS sub_ride_cnt,
        SUM(ride_cnt) AS total_ride_cnt
    FROM stg_ride_month
    WHERE month_ym BETWEEN 202310 AND 202512
    GROUP BY month_ym, month_date, gender, age_band
),
keys AS (
    SELECT month_ym, month_date, gender, age_band FROM signup
    UNION
    SELECT month_ym, month_date, gender, age_band FROM ride
),
base AS (
    SELECT
        k.month_ym,
        k.month_date,
        k.gender,
        k.age_band,
        COALESCE(s.signup_cnt, 0) AS signup_cnt,
        COALESCE(r.sub_ride_cnt, 0) AS sub_ride_cnt,
        COALESCE(r.total_ride_cnt, 0) AS total_ride_cnt
    FROM keys k
    LEFT JOIN signup s
        ON k.month_ym = s.month_ym
       AND k.gender = s.gender
       AND k.age_band = s.age_band
    LEFT JOIN ride r
        ON k.month_ym = r.month_ym
       AND k.gender = r.gender
       AND k.age_band = r.age_band
),
with_window AS (
    SELECT
        *,
        LAG(sub_ride_cnt) OVER (
            PARTITION BY gender, age_band
            ORDER BY month_ym
        ) AS prev_sub_ride_cnt,
        AVG(sub_ride_cnt) OVER (
            PARTITION BY gender, age_band
            ORDER BY month_ym
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS prev_3m_avg_sub_ride_cnt
    FROM base
)
SELECT
    month_ym,
    month_date,
    gender,
    age_band,
    signup_cnt,
    sub_ride_cnt,
    total_ride_cnt,
    CASE WHEN signup_cnt > 0 THEN 1.0 * sub_ride_cnt / signup_cnt ELSE NULL END AS activation_proxy,
    CASE WHEN total_ride_cnt > 0 THEN 1.0 * sub_ride_cnt / total_ride_cnt ELSE NULL END AS subscription_share,
    CASE WHEN prev_sub_ride_cnt > 0 THEN 1.0 * sub_ride_cnt / prev_sub_ride_cnt ELSE NULL END AS retention_proxy_mom,
    CASE WHEN prev_3m_avg_sub_ride_cnt > 0 THEN 1.0 * sub_ride_cnt / prev_3m_avg_sub_ride_cnt ELSE NULL END AS rolling_3m_persistence
FROM with_window
WHERE month_ym BETWEEN 202401 AND 202512;

CREATE TABLE mart_station_efficiency_month AS
WITH ride_station AS (
    SELECT
        month_ym,
        month_date,
        station_id,
        MAX(station_name) AS station_name_from_ride,
        SUM(ride_cnt) AS total_ride_cnt,
        SUM(CASE WHEN pass_group = 'SUBSCRIPTION' THEN ride_cnt ELSE 0 END) AS sub_ride_cnt,
        SUM(CASE WHEN pass_group = 'NON_SUBSCRIPTION' THEN ride_cnt ELSE 0 END) AS non_sub_ride_cnt,
        SUM(distance_m) AS total_distance_m,
        SUM(duration_min) AS total_duration_min
    FROM stg_ride_month
    WHERE month_ym BETWEEN 202401 AND 202512
      AND station_id IS NOT NULL
      AND is_ops_excluded = 0
    GROUP BY month_ym, month_date, station_id
),
joined AS (
    SELECT
        r.month_ym,
        r.month_date,
        m.snapshot_ym,
        r.station_id,
        COALESCE(s.station_name, r.station_name_from_ride) AS station_name,
        s.district,
        s.lat,
        s.lon,
        s.rack_cnt,
        r.total_ride_cnt,
        r.sub_ride_cnt,
        r.non_sub_ride_cnt,
        r.total_distance_m,
        r.total_duration_min
    FROM ride_station r
    LEFT JOIN stg_month_station_snapshot_map m
        ON r.month_ym = m.month_ym
    LEFT JOIN stg_station_snapshot s
        ON m.snapshot_ym = s.snapshot_ym
       AND r.station_id = s.station_id
)
SELECT
    month_ym,
    month_date,
    snapshot_ym,
    station_id,
    station_name,
    district,
    lat,
    lon,
    rack_cnt,
    total_ride_cnt,
    sub_ride_cnt,
    non_sub_ride_cnt,
    CASE WHEN total_ride_cnt > 0 THEN 1.0 * sub_ride_cnt / total_ride_cnt ELSE NULL END AS subscription_share,
    CASE WHEN rack_cnt > 0 THEN 1.0 * total_ride_cnt / rack_cnt ELSE NULL END AS rides_per_rack,
    total_distance_m,
    total_duration_min,
    CASE WHEN total_ride_cnt > 0 THEN 1.0 * total_distance_m / total_ride_cnt ELSE NULL END AS avg_distance_m_per_ride,
    CASE WHEN total_ride_cnt > 0 THEN 1.0 * total_duration_min / total_ride_cnt ELSE NULL END AS avg_duration_min_per_ride,
    CASE WHEN rack_cnt IS NULL THEN 0 ELSE 1 END AS station_snapshot_matched
FROM joined;

CREATE INDEX IF NOT EXISTS idx_mart_signup_segment_month_keys
ON mart_signup_segment_month(month_ym, gender, age_band);

CREATE INDEX IF NOT EXISTS idx_mart_ride_segment_month_keys
ON mart_ride_segment_month(month_ym, gender, age_band, ride_type);

CREATE INDEX IF NOT EXISTS idx_mart_growth_month_keys
ON mart_growth_month(month_ym, gender, age_band);

CREATE INDEX IF NOT EXISTS idx_mart_station_efficiency_month_keys
ON mart_station_efficiency_month(month_ym, station_id);

CREATE INDEX IF NOT EXISTS idx_mart_station_efficiency_district
ON mart_station_efficiency_month(month_ym, district);
'''
(SQL_DIR / "04_create_mart_tables.sql").write_text(sql, encoding="utf-8")
conn.executescript(sql)
conn.commit()
print("mart 테이블 생성 완료")

mart 테이블 생성 완료


In [7]:
mart_tables = [
    "mart_signup_segment_month",
    "mart_ride_segment_month",
    "mart_growth_month",
    "mart_station_efficiency_month"
]

mart_row_count_df = pd.DataFrame([
    {
        "table": table,
        "row_count": pd.read_sql_query(f"SELECT COUNT(*) AS c FROM {table}", conn).iloc[0, 0]
    }
    for table in mart_tables
])

mart_row_count_df

,table,row_count
0,mart_signup_segment_month,385
1,mart_ride_segment_month,1152
2,mart_growth_month,577
3,mart_station_efficiency_month,65744


In [8]:
mart_coverage_df = pd.concat([
    pd.read_sql_query(
        f'''
        SELECT
            '{table}' AS table_name,
            MIN(month_ym) AS min_ym,
            MAX(month_ym) AS max_ym,
            COUNT(DISTINCT month_ym) AS month_count
        FROM {table}
        ''',
        conn
    )
    for table in mart_tables
], ignore_index=True)

mart_coverage_df

,table_name,min_ym,max_ym,month_count
0,mart_signup_segment_month,202401,202512,24
1,mart_ride_segment_month,202401,202512,24
2,mart_growth_month,202401,202512,24
3,mart_station_efficiency_month,202401,202512,24


In [9]:
growth_metric_check_df = pd.read_sql_query(
    '''
    SELECT
        COUNT(*) AS row_count,
        SUM(CASE WHEN signup_cnt < 0 THEN 1 ELSE 0 END) AS negative_signup_cnt,
        SUM(CASE WHEN sub_ride_cnt < 0 THEN 1 ELSE 0 END) AS negative_sub_ride_cnt,
        SUM(CASE WHEN total_ride_cnt < 0 THEN 1 ELSE 0 END) AS negative_total_ride_cnt,
        SUM(CASE WHEN activation_proxy IS NULL THEN 1 ELSE 0 END) AS null_activation_proxy,
        SUM(CASE WHEN subscription_share IS NULL THEN 1 ELSE 0 END) AS null_subscription_share,
        SUM(CASE WHEN subscription_share < 0 OR subscription_share > 1 THEN 1 ELSE 0 END) AS invalid_subscription_share
    FROM mart_growth_month
    ''',
    conn
)

growth_metric_check_df

,row_count,negative_signup_cnt,negative_sub_ride_cnt,negative_total_ride_cnt,null_activation_proxy,null_subscription_share,invalid_subscription_share
0,577,0,0,0,192,1,0


In [10]:
station_join_check_df = pd.read_sql_query(
    '''
    SELECT
        month_ym,
        COUNT(*) AS station_month_count,
        SUM(station_snapshot_matched) AS matched_count,
        ROUND(100.0 * SUM(station_snapshot_matched) / COUNT(*), 2) AS matched_rate_pct,
        SUM(CASE WHEN rack_cnt IS NULL THEN 1 ELSE 0 END) AS null_rack_cnt_count,
        SUM(CASE WHEN rack_cnt <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_rack_cnt_count
    FROM mart_station_efficiency_month
    GROUP BY month_ym
    ORDER BY month_ym
    ''',
    conn
)

station_join_check_df

,month_ym,station_month_count,matched_count,matched_rate_pct,null_rack_cnt_count,zero_or_negative_rack_cnt_count
0,202401,2728,2728,100.00,0,0
1,202402,2727,2727,100.00,0,0
2,202403,2729,2724,99.82,5,0
3,202404,2730,2719,99.60,11,0
4,202405,2732,2724,99.71,8,0
5,202406,2734,2729,99.82,5,0
6,202407,2733,2724,99.67,9,0
7,202408,2722,2704,99.34,18,0
8,202409,2734,2713,99.23,21,0
9,202410,2743,2723,99.27,20,0


In [11]:
def df_to_md(df):
    if df is None or df.empty:
        return "_데이터 없음_"

    temp = df.copy().fillna("").astype(str)

    def clean_cell(x):
        return str(x).replace("\n", "<br>").replace("|", "\\|")

    headers = [clean_cell(c) for c in temp.columns]
    rows = [[clean_cell(v) for v in row] for row in temp.values.tolist()]
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

growth_sample_df = pd.read_sql_query(
    "SELECT * FROM mart_growth_month ORDER BY month_ym, gender, age_band LIMIT 20",
    conn
)

station_top_sample_df = pd.read_sql_query(
    """
    SELECT
        month_ym,
        station_id,
        station_name,
        district,
        total_ride_cnt,
        rack_cnt,
        ROUND(rides_per_rack, 2) AS rides_per_rack,
        ROUND(subscription_share, 3) AS subscription_share
    FROM mart_station_efficiency_month
    WHERE rack_cnt > 0
    ORDER BY rides_per_rack DESC
    LIMIT 20
    """,
    conn
)

summary_md = f"""# mart 생성 및 품질 점검 결과 요약

## 1. mart 테이블 행 수

{df_to_md(mart_row_count_df)}

## 2. mart 월별 커버리지

{df_to_md(mart_coverage_df)}

## 3. growth mart 핵심지표 점검

{df_to_md(growth_metric_check_df)}

## 4. station efficiency mart 조인 점검

{df_to_md(station_join_check_df)}

## 5. growth mart 샘플

{df_to_md(growth_sample_df)}

## 6. station efficiency 상위 샘플

{df_to_md(station_top_sample_df)}

## 해석 메모

- `mart_growth_month`는 월 × 성별 × 연령대 기준의 성장 분석 mart이다.
- `activation_proxy`는 `sub_ride_cnt / signup_cnt`로 계산한 세그먼트 단위 proxy이다.
- `retention_proxy_mom`은 전월 대비 정기권 이용 지속지수이다.
- `rolling_3m_persistence`는 직전 3개월 평균 대비 정기권 이용 지속지수이다.
- `mart_station_efficiency_month`는 월 × 대여소번호 기준의 운영 효율 mart이다.
- `rides_per_rack`은 `total_ride_cnt / rack_cnt`로 계산한다.
- `station_snapshot_matched = 0`인 행은 대여소 스냅샷과 매칭되지 않은 경우이므로 운영 효율 해석에서 주의해야 한다.
"""

summary_path = REPORT_DIR / "mart_check_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")

print("mart 점검 요약 저장 완료:", summary_path)

mart 점검 요약 저장 완료: d:\seoul-bike-aarrr-analysis\reports\mart_check_summary.md


In [13]:
qc_sql_path = SQL_DIR / "05_mart_quality_check_queries.sql"
qc_sql_path.write_text(r'''-- 05_mart_quality_check_queries.sql
-- SQLite 기준 mart 품질 점검 쿼리

SELECT 'mart_signup_segment_month' AS table_name, COUNT(*) AS row_count FROM mart_signup_segment_month
UNION ALL
SELECT 'mart_ride_segment_month', COUNT(*) FROM mart_ride_segment_month
UNION ALL
SELECT 'mart_growth_month', COUNT(*) FROM mart_growth_month
UNION ALL
SELECT 'mart_station_efficiency_month', COUNT(*) FROM mart_station_efficiency_month;

SELECT 'mart_signup_segment_month' AS table_name, MIN(month_ym) AS min_ym, MAX(month_ym) AS max_ym, COUNT(DISTINCT month_ym) AS month_count FROM mart_signup_segment_month
UNION ALL
SELECT 'mart_ride_segment_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM mart_ride_segment_month
UNION ALL
SELECT 'mart_growth_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM mart_growth_month
UNION ALL
SELECT 'mart_station_efficiency_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM mart_station_efficiency_month;

SELECT
    COUNT(*) AS row_count,
    SUM(CASE WHEN signup_cnt < 0 THEN 1 ELSE 0 END) AS negative_signup_cnt,
    SUM(CASE WHEN sub_ride_cnt < 0 THEN 1 ELSE 0 END) AS negative_sub_ride_cnt,
    SUM(CASE WHEN total_ride_cnt < 0 THEN 1 ELSE 0 END) AS negative_total_ride_cnt,
    SUM(CASE WHEN activation_proxy IS NULL THEN 1 ELSE 0 END) AS null_activation_proxy,
    SUM(CASE WHEN subscription_share IS NULL THEN 1 ELSE 0 END) AS null_subscription_share,
    SUM(CASE WHEN subscription_share < 0 OR subscription_share > 1 THEN 1 ELSE 0 END) AS invalid_subscription_share
FROM mart_growth_month;

SELECT
    month_ym,
    COUNT(*) AS station_month_count,
    SUM(station_snapshot_matched) AS matched_count,
    ROUND(100.0 * SUM(station_snapshot_matched) / COUNT(*), 2) AS matched_rate_pct,
    SUM(CASE WHEN rack_cnt IS NULL THEN 1 ELSE 0 END) AS null_rack_cnt_count,
    SUM(CASE WHEN rack_cnt <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_rack_cnt_count
FROM mart_station_efficiency_month
GROUP BY month_ym
ORDER BY month_ym;

SELECT
    month_ym,
    station_id,
    station_name,
    district,
    total_ride_cnt,
    rack_cnt,
    ROUND(rides_per_rack, 2) AS rides_per_rack,
    ROUND(subscription_share, 3) AS subscription_share
FROM mart_station_efficiency_month
WHERE rack_cnt > 0
ORDER BY rides_per_rack DESC
LIMIT 20;
''', encoding="utf-8")
print("mart 품질 점검 SQL 저장:", qc_sql_path)

mart 품질 점검 SQL 저장: d:\seoul-bike-aarrr-analysis\sql\05_mart_quality_check_queries.sql


In [ ]:
conn.close()
print('SQLite 연결 종료 완료')